# CLIP Ver2: Prompt-Based Scene dan Objek Wisata

Notebook ini memakai CLIP untuk memilih deskripsi prompt yang paling cocok dengan setiap gambar di `dataset/images_skema2`. Output berfokus pada tiga sinyal utama: label 4A, scene, dan objek wisata.

Catatan penting: CLIP bukan model generatif, jadi CLIP tidak bisa menulis jawaban bebas dari gambar. Versi ini tidak memakai `LABELS` dictionary panjang seperti notebook sebelumnya, tetapi tetap memakai bank prompt ringkas agar CLIP punya opsi teks untuk dibandingkan dengan gambar.

Jika package belum tersedia, jalankan sekali:

```python
# %pip install -q torch transformers pillow pandas
```

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
import re

import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


def find_existing_path(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f"Tidak ada path yang ditemukan dari kandidat: {candidates}")


DATASET_XLSX = find_existing_path("dataset/labels_skema2.xlsx", "../dataset/labels_skema2.xlsx")
IMAGE_DIR = find_existing_path("dataset/images_skema2", "../dataset/images_skema2")
OUTPUT_CSV = DATASET_XLSX.parent / "clip_ver2.csv"

MODEL_NAME = "openai/clip-vit-base-patch32"
LOCAL_FILES_ONLY = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
OUTPUT_COLUMNS = ["ClipV2_4A", "ClipV2_Scene", "ClipV2_ObjekWisata"]

print(f"Dataset: {DATASET_XLSX}")
print(f"Folder gambar: {IMAGE_DIR}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Device: {DEVICE}")

In [ ]:
XLSX_NS = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
COLUMN_RE = re.compile(r"([A-Z]+)")
BUILTIN_DATE_FORMAT_IDS = {14, 15, 16, 17, 22, 27, 30, 36, 45, 46, 47, 50, 57}


def column_ref_to_index(cell_ref: str) -> int:
    match = COLUMN_RE.match(cell_ref)
    if not match:
        raise ValueError(f"Referensi cell tidak valid: {cell_ref}")

    index = 0
    for char in match.group(1):
        index = index * 26 + (ord(char) - ord("A") + 1)
    return index - 1


def excel_serial_to_datetime_text(value: str) -> str:
    try:
        total_seconds = round(float(value) * 86400)
    except (TypeError, ValueError):
        return value

    parsed = datetime(1899, 12, 30) + timedelta(seconds=total_seconds)
    return parsed.strftime("%Y-%m-%d %H:%M:%S")


def load_date_style_ids(zip_file: ZipFile) -> set[str]:
    if "xl/styles.xml" not in zip_file.namelist():
        return set()

    styles = ET.fromstring(zip_file.read("xl/styles.xml"))
    custom_formats = {}
    num_fmts = styles.find("a:numFmts", XLSX_NS)
    if num_fmts is not None:
        for num_fmt in num_fmts.findall("a:numFmt", XLSX_NS):
            custom_formats[int(num_fmt.attrib["numFmtId"])] = num_fmt.attrib.get("formatCode", "").lower()

    date_style_ids = set()
    cell_xfs = styles.find("a:cellXfs", XLSX_NS)
    if cell_xfs is None:
        return date_style_ids

    for style_index, xf in enumerate(cell_xfs.findall("a:xf", XLSX_NS)):
        num_fmt_id = int(xf.attrib.get("numFmtId", 0))
        format_code = custom_formats.get(num_fmt_id, "")
        is_builtin_date = num_fmt_id in BUILTIN_DATE_FORMAT_IDS
        is_custom_date = any(token in format_code for token in ["yy", "dd", "hh"])
        if is_builtin_date or is_custom_date:
            date_style_ids.add(str(style_index))

    return date_style_ids


def load_shared_strings(zip_file: ZipFile) -> list[str]:
    if "xl/sharedStrings.xml" not in zip_file.namelist():
        return []

    shared_root = ET.fromstring(zip_file.read("xl/sharedStrings.xml"))
    shared_strings = []
    for item in shared_root.findall("a:si", XLSX_NS):
        shared_strings.append("".join(text.text or "" for text in item.findall(".//a:t", XLSX_NS)))
    return shared_strings


def read_xlsx_first_sheet(path: Path) -> pd.DataFrame:
    with ZipFile(path) as zip_file:
        shared_strings = load_shared_strings(zip_file)
        date_style_ids = load_date_style_ids(zip_file)
        sheet = ET.fromstring(zip_file.read("xl/worksheets/sheet1.xml"))

        row_maps = []
        max_column_count = 0
        for row in sheet.findall(".//a:sheetData/a:row", XLSX_NS):
            row_values = {}
            for cell in row.findall("a:c", XLSX_NS):
                column_index = column_ref_to_index(cell.attrib["r"])
                max_column_count = max(max_column_count, column_index + 1)
                cell_type = cell.attrib.get("t")

                if cell_type == "inlineStr":
                    value = "".join(text.text or "" for text in cell.findall(".//a:t", XLSX_NS))
                else:
                    value_node = cell.find("a:v", XLSX_NS)
                    if value_node is None:
                        value = ""
                    elif cell_type == "s":
                        value = shared_strings[int(value_node.text)]
                    else:
                        value = value_node.text or ""
                        if cell.attrib.get("s") in date_style_ids:
                            value = excel_serial_to_datetime_text(value)

                row_values[column_index] = value
            row_maps.append(row_values)

        parsed_rows = [
            [row_values.get(index, "") for index in range(max_column_count)]
            for row_values in row_maps
        ]

    headers = parsed_rows[0]
    rows = parsed_rows[1:]
    while headers and headers[-1] == "":
        headers.pop()
        rows = [row[:-1] for row in rows]

    return pd.DataFrame(rows, columns=headers)


df = read_xlsx_first_sheet(DATASET_XLSX)
display(df.head())
print(f"Jumlah baris label: {len(df)}")
print("Kolom:", list(df.columns))

In [ ]:
if "File" not in df.columns:
    raise KeyError("Kolom 'File' tidak ditemukan di labels_skema2.xlsx")
if "Catatan" not in df.columns:
    raise KeyError("Kolom 'Catatan' tidak ditemukan di labels_skema2.xlsx")

file_series = df["File"].astype("string").fillna("").str.strip()
files_from_excel = [file_name for file_name in file_series.tolist() if file_name]
image_paths_by_name = {
    path.name: path
    for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
}

missing_images = sorted(set(files_from_excel) - set(image_paths_by_name))
extra_images = sorted(set(image_paths_by_name) - set(files_from_excel))

print(f"Baris dengan File terisi: {len(files_from_excel)}")
print(f"Gambar di folder images_skema2: {len(image_paths_by_name)}")
print(f"File dari Excel yang tidak ada di folder: {len(missing_images)}")
print(f"Gambar di folder yang tidak ada di Excel: {len(extra_images)}")

if missing_images:
    display(pd.DataFrame({"missing_images": missing_images}))
    raise FileNotFoundError("Ada nama file di kolom 'File' yang tidak ditemukan di folder images_skema2.")

if extra_images:
    display(pd.DataFrame({"extra_images": extra_images}))

ordered_image_paths = [image_paths_by_name[file_name] for file_name in files_from_excel]

In [ ]:
try:
    processor = CLIPProcessor.from_pretrained(
        MODEL_NAME,
        local_files_only=LOCAL_FILES_ONLY,
        use_fast=False,
    )
    model = CLIPModel.from_pretrained(
        MODEL_NAME,
        local_files_only=LOCAL_FILES_ONLY,
    ).to(DEVICE)
    model.eval()
except OSError as exc:
    raise RuntimeError(
        "Model CLIP belum ada di cache lokal. Jika ingin download otomatis, ubah "
        "LOCAL_FILES_ONLY = False lalu jalankan ulang cell ini dengan koneksi internet."
    ) from exc

print("Model CLIP siap dipakai.")

In [ ]:
four_a_prompts = pd.DataFrame(
    [
        (
            "Attraction",
            "a tourism photo mainly showing a destination attraction such as a beach, sea, island, mountain, waterfall, forest, landscape, cultural landmark, monument, temple, recreation area, or outdoor tourism activity",
        ),
        (
            "Accessibility",
            "a tourism photo mainly showing access or transportation such as a road, bridge, harbor, dock, boat, ferry, vehicle, parking area, path, stairs, entrance gate, or direction sign",
        ),
        (
            "Amenities",
            "a tourism photo mainly showing facilities or amenities such as a hotel, resort, restaurant, cafe, toilet, mosque, gazebo, seating area, swimming pool, playground, market, information board, or resting area",
        ),
        (
            "Ancillary",
            "a tourism photo mainly showing supporting services or activities such as a tourism event, festival, cultural performance, tour guide, tourism office, information center, promotion banner, security officer, ticket counter, or tourism management activity",
        ),
    ],
    columns=["label", "prompt"],
)

scene_prompts = pd.DataFrame(
    [
        ("pantai atau pesisir", "a beach or coastal tourism scene with sand, shoreline, waves, or sea view"),
        ("wisata bahari", "a marine tourism scene with ocean, island, boat, snorkeling, diving, or tropical water"),
        ("pegunungan atau dataran tinggi", "a mountain or highland tourism scene with hills, cliffs, volcano, or scenic altitude"),
        ("air terjun sungai atau danau", "a waterfall, river, lake, or freshwater nature tourism scene"),
        ("hutan atau alam hijau", "a forest, greenery, eco tourism, or tropical natural landscape scene"),
        ("budaya atau sejarah", "a cultural or historical tourism scene with heritage buildings, monument, temple, traditional house, or performance"),
        ("rekreasi keluarga", "a family recreation tourism scene with playground, swimming pool, amusement, gazebo, or leisure area"),
        ("fasilitas wisata", "a tourism facility scene with amenities, seating area, restaurant, cafe, hotel, resort, toilet, mosque, or information board"),
        ("akses atau transportasi", "a tourism access scene with road, bridge, vehicle, harbor, dock, ferry, stairs, pathway, entrance gate, or sign"),
        ("acara atau promosi wisata", "a tourism event or promotion scene with festival, cultural performance, banner, campaign, exhibition, or community activity"),
        ("perkotaan", "an urban tourism scene with city area, modern place, street, building, or public space"),
        ("pedesaan atau tradisional", "a rural or traditional tourism scene with village, local community, traditional environment, or local culture"),
        ("suasana santai", "a relaxing tourism atmosphere with calm place, leisure, vacation, rest, or peaceful scenery"),
        ("keramaian wisatawan", "a crowded tourism area with many tourists, visitors, or group activity"),
    ],
    columns=["label", "prompt"],
)

object_prompts = pd.DataFrame(
    [
        ("pantai", "a beach as the main tourism object"),
        ("laut", "the sea or ocean as the main tourism object"),
        ("pulau", "an island as the main tourism object"),
        ("gunung", "a mountain, hill, cliff, or volcano as the main tourism object"),
        ("air terjun", "a waterfall as the main tourism object"),
        ("sungai atau danau", "a river or lake as the main tourism object"),
        ("hutan atau lanskap alam", "forest, greenery, or natural landscape as the main tourism object"),
        ("bangunan budaya atau sejarah", "a cultural landmark, historical building, temple, monument, or traditional house as the main tourism object"),
        ("area rekreasi", "a recreation area, playground, swimming pool, camping area, or outdoor activity place as the main tourism object"),
        ("hotel atau resort", "a hotel, resort, homestay, or accommodation as the main tourism object"),
        ("restoran atau kafe", "a restaurant, cafe, food stall, market, or culinary facility as the main tourism object"),
        ("gazebo atau area duduk", "a gazebo, seating area, resting area, or visitor facility as the main tourism object"),
        ("jalan atau jalur", "a road, pathway, stairs, bridge, entrance gate, or direction sign as the main tourism object"),
        ("pelabuhan dermaga atau kapal", "a harbor, dock, boat, ferry, or marine transportation as the main tourism object"),
        ("kendaraan", "a vehicle, car, motorcycle, bus, or transportation as the main tourism object"),
        ("papan informasi atau signage", "an information board, tourism sign, direction sign, official signage, or promotion banner as the main tourism object"),
        ("festival atau pertunjukan", "a festival, event, cultural performance, exhibition, or tourism campaign as the main tourism object"),
        ("pemandu atau komunitas wisata", "a tour guide, local guide, tourism office, tourism organization, volunteers, or community activity as the main tourism object"),
        ("wisatawan", "tourists, visitors, crowd, or people doing tourism activities as the main tourism object"),
    ],
    columns=["label", "prompt"],
)

prompt_banks = [
    ("ClipV2_4A", four_a_prompts),
    ("ClipV2_Scene", scene_prompts),
    ("ClipV2_ObjekWisata", object_prompts),
]

for name, bank in prompt_banks:
    print(f"{name}: {len(bank)} prompt")

In [ ]:
@torch.inference_mode()
def build_text_features(prompt_bank: pd.DataFrame) -> torch.Tensor:
    inputs = processor(
        text=prompt_bank["prompt"].tolist(),
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(DEVICE)
    text_features = model.get_text_features(**inputs)
    return text_features / text_features.norm(dim=-1, keepdim=True)


def load_image(image_path: Path) -> Image.Image:
    with Image.open(image_path) as image:
        return image.convert("RGB").copy()


@torch.inference_mode()
def predict_clip_ver2_batch(image_paths: list[Path], prompt_banks: list[tuple[str, pd.DataFrame]], batch_size: int = BATCH_SIZE) -> dict[str, dict[str, str]]:
    bank_features = [
        (column_name, bank, build_text_features(bank))
        for column_name, bank in prompt_banks
    ]
    predictions = {}
    total = len(image_paths)

    for start in range(0, total, batch_size):
        batch_paths = image_paths[start : start + batch_size]
        valid_paths = []
        images = []

        for image_path in batch_paths:
            try:
                images.append(load_image(image_path))
                valid_paths.append(image_path)
            except Exception as exc:
                print(f"Gagal memproses {image_path.name}: {exc}")

        if not images:
            continue

        inputs = processor(images=images, return_tensors="pt").to(DEVICE)
        image_features = model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        batch_predictions = [{column: "" for column in OUTPUT_COLUMNS} for _ in valid_paths]
        for column_name, bank, text_features in bank_features:
            similarities = image_features @ text_features.T
            best_indices = similarities.argmax(dim=1).cpu().tolist()
            for prediction, best_index in zip(batch_predictions, best_indices):
                prediction[column_name] = bank.iloc[best_index]["label"]

        for image_path, prediction in zip(valid_paths, batch_predictions):
            predictions[image_path.name] = prediction

        done = min(start + len(batch_paths), total)
        print(f"Progress CLIP Ver2: {done}/{total} gambar", end="\r")

    print(f"Progress CLIP Ver2: {total}/{total} gambar")
    return predictions

In [ ]:
clip_ver2_predictions = predict_clip_ver2_batch(ordered_image_paths, prompt_banks)

result_df = df.copy()
for column in OUTPUT_COLUMNS:
    result_df[column] = ""

for row_index, file_name in file_series.items():
    if not file_name:
        continue

    prediction = clip_ver2_predictions.get(file_name, {})
    for column in OUTPUT_COLUMNS:
        result_df.at[row_index, column] = prediction.get(column, "")

base_columns = list(df.columns)
insert_position = base_columns.index("Catatan") + 1
ordered_columns = base_columns[:insert_position] + OUTPUT_COLUMNS + base_columns[insert_position:]
result_df = result_df[ordered_columns]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"CSV berhasil disimpan: {OUTPUT_CSV}")
print(f"Jumlah baris output: {len(result_df)}")
display(result_df.head())